In [3]:
# Need to take a head, relation, tail (h,r,t) triple
# Need to store it on the graph
# Heads and tails are not embeddings. They are simply nodes
# each node has a head_key and a tail_query. the relation is in fact computed at runtime from head_key.tail_query^T
# if a relation is positive, it is navigable.
# if many relations are positive, the largest one is the one that is chosen. If many are large, a random one is chosen.
# When we are trying to combine a relational composite, we instead matmul - composite node key = head_key x tail_query^T
# In this way, we can build an ongoing context. The details of this need to be ironed out. The other options include an ever growing matrix
# of prior relational composites or even just a growing list of head_keys that are dotproducted against each tail_query as you go.
# This has the disadvantage of long chains of thought growing more expensive but obviously it provides the ability to retain early context
# The weights of head_key and tail_query for each node are randomly initialised and the backpropagated with a high learning rate
# when given evidence.
# The question remains about how to make a given relation interpretable.
# If we take a statement "Dogs are mammals" we can decompose into either `Dogs -are> mammals` or `Dogs -> are -> mammals`.
# To me intuitively, the second seems cleaner

In [2]:
import torch
import torch.nn.functional as F

dim = 8

In [12]:
dogs_k = torch.randn(dim, dim)
dogs_q = torch.randn(dim, dim)
are_k = torch.randn(dim, dim)
are_q = torch.randn(dim, dim)
mammals_k = torch.randn(dim, dim)
mammals_q = torch.randn(dim, dim)
have_k = torch.randn(dim, dim)
have_q = torch.randn(dim, dim)
teeth_k = torch.randn(dim, dim)
teeth_q = torch.randn(dim, dim)
produce_k = torch.randn(dim, dim)
produce_q = torch.randn(dim, dim)
milk_k = torch.randn(dim, dim)
milk_q = torch.randn(dim, dim)
reptiles_k = torch.randn(dim, dim)
reptiles_q = torch.randn(dim, dim)

# so now, if deciding where to navigate from dogs, we should matmul to identify the valid egress pathways

dogs_are = dogs_k @ are_q

dogs_are_mammals = dogs_are @ mammals_q
print(dogs_are_mammals[0, 0]) # should be > 0

mammals_produce = mammals_k @ produce_q
mammals_produce_milk = mammals_produce @ milk_q
print(mammals_produce_milk[0, 0]) # should be > 0

dogs_produce = dogs_k @ produce_q
dogs_produce_milk = dogs_produce @ milk_q
print(dogs_produce_milk[0, 0]) # should be > 0

dogs_are_reptiles = dogs_are @ reptiles_q
print(dogs_are_reptiles[0, 0]) # should be < 0

tensor(4.6237)
tensor(-8.4457)
tensor(-10.6934)
tensor(4.7885)


In [23]:
knowledge = {}
statement_history = {}

def memorise_statement(key: str, value: str, label: bool, key_index: int):
    if key not in statement_history:
        statement_history[key] = {}
    statement_history[key][value] = {
        "label": label,
        "index": key_index
    }
    
def new_node(value: str):
    return {
        "key": torch.randn(dim, dim, requires_grad=True),
        "query": torch.randn(dim, dim, requires_grad=True),
        "value": value,
    }

def learn(statement: str, label: bool):
    print("learning " + statement)
    words = statement.lower().split(" ")
    for i, word in enumerate(words):
        if word not in knowledge:
            knowledge[word] = new_node(word)
        if i > 0:
            memorise_statement(word, statement, label, i)

    training_set = []
    for word in words[1:]:
        training_set.append(statement_history[word])
    
    
    labels = []
    keys_used = set()
    queries_used = set()

    raw_scores = []
    for training_item in training_set:
        for statement, metadata in training_item.items():
            words = statement.lower().split(" ")
            key = knowledge[words[0]]["key"]
            keys_used.add(words[0])
            query = knowledge[words[1]]["query"]
            queries_used.add(words[1])
            composite = key @ query
            score = composite[0, 0]
            raw_scores.append(score)
            labels.append(metadata["label"])
            for i in range(2, metadata["index"] + 1):
                queries_used.add(words[i])
                query = knowledge[words[i]]["query"]
                composite = composite @ query
                score = composite[0, 0]
                raw_scores.append(score)
                labels.append(metadata["label"])

    # the goal of the training is to take the actual raw dot products (the scores) of each of the transitions and use that as the 
    # source of our cross-entropy used for training our parameters. We need to do not only this, but grab all the out_edges from each of our
    # words and include their raw dot products as well as their logits in the training to avoid catatstrophic forgetting

    params = [knowledge[w]["key"] for w in keys_used] + [knowledge[w]["query"] for w in queries_used]

    
    labels = torch.tensor([float(l) for l in labels])
    scores = torch.stack(raw_scores)

    print("scores: ", scores)
    print("labels: ", labels)
    optimiser = torch.optim.Adam(params, lr=0.1)
    print("training set: ", training_set)
    
    for _ in range(50):
        optimiser.zero_grad()
        raw_scores = []
        for training_item in training_set:
            for statement, metadata in training_item.items():
                words = statement.lower().split(" ")
                key = knowledge[words[0]]["key"]
                query = knowledge[words[1]]["query"]
                composite = key @ query
                score = composite[0, 0]
                raw_scores.append(score)
                for i in range(2, metadata["index"] + 1):
                    query = knowledge[words[i]]["query"]
                    composite = composite @ query
                    score = composite[0, 0]
                    raw_scores.append(score)
        scores = torch.stack(raw_scores)
        losses = F.binary_cross_entropy_with_logits(scores, labels, reduction="none")
        loss = losses.mean()
        loss.backward()
        optimiser.step()
        print("losses: ", losses)
    # here we train with high learning rate. head_tail should be 1. 

learn("Sandwiches are green", False)
learn("Dogs are mammals", True)
learn("Dogs are reptiles", False)
learn("Mammals produce milk", True)
learn("reptiles produce milk", False)
learn("Snakes are reptiles", True)
learn("parrots are multi-coloured", False)

learning Sandwiches are green
scores:  tensor([0.1340, 0.1340, 6.2694], grad_fn=<StackBackward0>)
labels:  tensor([0., 0., 0.])
training set:  [{'Sandwiches are green': {'label': False, 'index': 1}}, {'Sandwiches are green': {'label': False, 'index': 2}}]
losses:  tensor([0.7624, 0.7624, 6.2713],
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
losses:  tensor([0.4819, 0.4819, 0.0022],
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
losses:  tensor([2.4133e-01, 2.4133e-01, 1.9073e-06],
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
losses:  tensor([0.1156, 0.1156, 0.0000],
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
losses:  tensor([0.0574, 0.0574, 0.0000],
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
losses:  tensor([0.0304, 0.0304, 0.0000],
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
losses:  tensor([0.0173, 0.0173, 0.0000],
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
losses:  tensor([0.0105, 0.0105, 0.0000],
    

In [49]:
def check(statement: str) -> bool:
    words = statement.lower().split(" ")
    if len(words) < 2: 
        print(statement + ": ", True)
        
    nodes = [knowledge[word] for word in words]
    composite = nodes[0]['key'] @ nodes[1]['query']
    
    for i in range(2, len(nodes)):
        composite = composite @ nodes[i]['query']

    print(statement + ": ", (composite[0, 0] >= 0).item())

In [50]:
check("Sandwiches are green")
check("Dogs are mammals")
check("Dogs are reptiles")
check("Mammals produce milk")
check("reptiles produce milk")
check("Snakes are reptiles")
check("parrots are multi-coloured")

Sandwiches are green:  False
Dogs are mammals:  True
Dogs are reptiles:  False
Mammals produce milk:  True
reptiles produce milk:  False
Snakes are reptiles:  False
parrots are multi-coloured:  False
